# Ablation 3 Counterfactual: Single-Scale + RandAugment (Smoke Test)

**Research Question**: Open: does single-scale training (512x512) with RandAugment avoid the 95% collapse seen in v5.

**Intervention**: Train with single resolution (imgsz=512) and explicit RandAugment, all other augmentations default.

**Hypothesis**: the v5 collapse came from multi-scale conflicting with RandAugment, not from RandAugment.

**Expected Outcome**: Dice ≥75% (proving RandAugment works with single-scale)

**Status**: Smoke test (1 seed, 10 epochs, tiny dataset, CPU)


In [ ]:
# Lock NumPy 1.26.4 (ultralytics upgrades to 2.x, breaks OpenCV)
import subprocess
import sys

print("Installing ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

print("Force-reinstalling NumPy 1.26.4...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy==1.26.4"
], check=True)

import numpy as np
print(f"OK NumPy locked: {np.__version__}")
assert np.__version__.startswith("1.26"), f"NumPy version mismatch: {np.__version__}"


In [ ]:
import os
from pathlib import Path
import shutil

# Kaggle paths
IN_KAGGLE = os.path.exists("/kaggle/input")
BASE_INPUT_PATH = Path("/kaggle/input") if IN_KAGGLE else Path(".")
OUTPUT_PATH = Path("/kaggle/working") if IN_KAGGLE else Path(".")

# Dataset paths (TINY for smoke test)
DATASET_PATH = BASE_INPUT_PATH / "tiny-wound-seg-smoke-test"
CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"

# Output directory
ABLATION_OUTPUT = OUTPUT_PATH / "ablation_3_counterfactual_smoke"
ABLATION_OUTPUT.mkdir(exist_ok=True, parents=True)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset: {DATASET_PATH}")
print(f"Code: {CODE_PATH}")
print(f"Output: {ABLATION_OUTPUT}")


In [ ]:
# Copy tiny dataset to working directory
# Tiny dataset structure: train/images, train/masks, test/images, test/masks
WORKING_DATASET = OUTPUT_PATH

# Copy train and test directories
for subdir in ["train", "test"]:
    src = DATASET_PATH / subdir
    dst = WORKING_DATASET / subdir
    if not dst.exists():
        print(f"Copying {subdir} from {src}...")
        shutil.copytree(src, dst)
        print(f"OK {subdir} copied to {dst}")
    else:
        print(f"OK {subdir} already exists at {dst}")

# Verify structure
print("\nDataset structure verification:")
for split in ["train", "test"]:
    for subdir in ["images", "masks", "labels"]:
        path = WORKING_DATASET / split / subdir
        count = len(list(path.glob("*"))) if path.exists() else 0
        print(f"  {split}/{subdir}: {count} files")


In [ ]:
import sys
sys.path.insert(0, str(CODE_PATH))

print(f"OK Added {CODE_PATH} to Python path")
print(f"sys.path[0]: {sys.path[0]}")


In [ ]:
from ultralytics import YOLO
import yaml
import json
from datetime import datetime

print("OK Imports successful")


In [ ]:
# Create YOLO dataset configuration for tiny dataset
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_content = f"""path: {OUTPUT_PATH}
train: train
val: test

names:
  0: wound
"""

with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"OK YOLO config created: {config_path}")

# Verify config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    print(f"  Dataset path: {config['path']}")
    print(f"  Train: {config['train']}")
    print(f"  Val: {config['val']}")


In [ ]:
# Training configuration
RANDOM_SEED = 42  # Single seed for smoke test
EPOCHS = 10        # Quick validation
PATIENCE = 5

# P100-optimized hyperparameters
BATCH = 32
LR0 = 0.015
WORKERS = 8
AMP = False

# COUNTERFACTUAL: Single-scale + RandAugment
IMGSZ = 512  # Single resolution (NOT multi-scale)
AUTO_AUGMENT = "randaugment"  # Explicit RandAugment

print("Training Configuration (Counterfactual):")
print(f"  Seed: {RANDOM_SEED}")
print(f"  Epochs: {EPOCHS} (patience: {PATIENCE})")
print(f"  Batch: {BATCH}")
print(f"  Learning rate: {LR0}")
print(f"  Workers: {WORKERS}")
print(f"  AMP: {AMP}")
print(f"  Image size: {IMGSZ} (SINGLE-SCALE)")
print(f"  Auto-augment: {AUTO_AUGMENT} (RandAugment)")


In [ ]:
# Training with single-scale + RandAugment
print(f"\n{'='*60}")
print(f"Training counterfactual (single-scale + RandAugment)")
print(f"{'='*60}")

# Initialize model
model = YOLO("yolov8n-seg.pt")

# Train with MINIMAL parameters (let YOLOv8 use defaults for other augmentations)
results = model.train(
    data=str(config_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,  # Single-scale: 512
    batch=BATCH,
    lr0=LR0,
    workers=WORKERS,
    patience=PATIENCE,
    amp=AMP,
    seed=RANDOM_SEED,
    save=True,
    project=str(ABLATION_OUTPUT / "yolo_training"),
    name=f"seed_{RANDOM_SEED}",
    exist_ok=True,
    verbose=False,

    # ONLY specify the intervention parameter
    auto_augment=AUTO_AUGMENT,  # Explicit RandAugment
    # All other augmentations use YOLOv8 defaults
)

# Save best model path
best_model_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{RANDOM_SEED}" / "weights" / "best.pt"
print(f"OK Training complete: {best_model_path}")


## args.yaml Verification

**Check**: YOLOv8 used single-scale with RandAugment.


In [ ]:
import yaml

print("\n=== PARAMETER VERIFICATION ===\n")

args_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{RANDOM_SEED}" / "args.yaml"

with open(args_path, 'r') as f:
    args = yaml.safe_load(f)

print(f"Checking critical parameters:")

# Verify single-scale
imgsz = args.get('imgsz')
print(f"  imgsz: {imgsz}")
if imgsz != 512:
    print(f"    WARNING  Expected 512, got {imgsz}")
else:
    print(f"    OK Single-scale correctly applied")

# Verify RandAugment
auto_augment = args.get('auto_augment')
print(f"  auto_augment: {auto_augment}")
if auto_augment != "randaugment":
    print(f"    WARNING  Expected 'randaugment', got {auto_augment}")
else:
    print(f"    OK RandAugment correctly applied")

# Show other augmentation parameters (should be YOLOv8 defaults)
print(f"\nOther augmentation parameters (YOLOv8 defaults):")
aug_params = ['hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale',
              'fliplr', 'mosaic', 'mixup', 'copy_paste', 'erasing']
for param in aug_params:
    value = args.get(param, 'not set')
    print(f"  {param}: {value}")


In [ ]:
# Save completion status
completion_status = {
    "notebook": "ablation_3_counterfactual_smoke",
    "status": "complete",
    "timestamp": datetime.now().isoformat(),
    "seed": RANDOM_SEED,
    "configuration": {
        "intervention": "Single-scale (512) + RandAugment",
        "imgsz": IMGSZ,
        "auto_augment": AUTO_AUGMENT,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "batch": BATCH,
        "lr0": LR0,
        "workers": WORKERS,
        "amp": AMP,
        "dataset": "tiny-wound-seg-smoke-test"
    },
    "best_model": str(best_model_path)
}

status_path = ABLATION_OUTPUT / "completion_status.json"
with open(status_path, 'w') as f:
    json.dump(completion_status, f, indent=2)

print(f"OK Completion status saved: {status_path}")
print(f"\nSmoke test complete!")
print(f"Next: Check Dice score to validate hypothesis")
